### **Remote Path**

In [1]:
import sys

sys.path.append('/home/dyuser-e/NI_project_dyuser-e_2026/dynap-se1')


### **Local Path**

In [ ]:
import sys
from pathlib import Path

dynapse_path = Path.home() / "NI_project" / "dynap-se1"
sys.path.append(str(dynapse_path))


### **Imports**

In [2]:

import samna
import numpy as np
import matplotlib.pyplot as plt
import samna.dynapse1 as dyn1

import dynapse1utils as ut
from netgen import Neuron, NetworkGenerator

from tools.spikegen_tools import get_instantaneous_activity
from tools.spikegen_tools import generate_regular_rate_spiketrain

from tools.dynapse_tuning_companion import DynapseTuningCompanion

import params
from params import set_params

import time
from IPython.display import Image

In [ ]:
import samna
import samna.dynapse1 as dyn1

# append the root folder to import from the repository
import sys
sys.path.append('..')

# essential constants for DYNAP-SE1
from dynapse1constants import *

# main DYNAP-SE1 toolkit
import dynapse1utils as ut

# Network Generator helper class
import netgen as n
from netgen import Neuron

# Other standard utilities
import time
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

### **Remote Connection**

In [ ]:
# get index for board (DONT USE TO CONNECT)

model, _ = ut.open_dynapse1(gui=False, select_device=True)


[0]:  Bus 1 Device 64 Dynapse1DevKit serial_number 00000026
[1]:  Bus 1 Device 81 Dynapse1DevKit serial_number 00000000
[2]:  Bus 1 Device 80 Dynapse1DevKit serial_number 00000033
[3]:  Bus 1 Device 95 Dynapse1DevKit serial_number 00000001
[4]:  Bus 1 Device 94 Dynapse1DevKit serial_number 00000007
[5]:  Bus 1 Device 93 Dynapse1DevKit serial_number 00000020


In [ ]:

model = ut.open_specific_device_in_sequence(3)

### **Local Connection**

In [ ]:
model, _ = ut.open_dynapse1()

In [ ]:
ut.open_gui(model)

In [ ]:
api = model.get_dynapse1_api()          # get Dynapse1 api from the model
set_params(model)                       # set standard parameters


config1     = model.get_configuration() 
param_group = config1.chips[0].cores[0].parameter_group 

In [ ]:
# create triplets of indices of neurons to monitor
chip_id=1
core_id=0

neurons_ids = np.arange(10, 20)

neurons_ids_triplets = [(chip_id, core_id, n_id) for n_id in neurons_ids]
event_graph, filter_node, sink_node = ut.create_neuron_select_graph(model, neurons_ids_triplets)

event_graph.start()

### **Initialize Spike Generator**

In [ ]:
# prepare the spike train that the spike generators will fire

# in our case it will be a sequence of spikes at a fixed rate:
spikegen_regular_rate = 100 #Hz

rates_vector = (spikegen_regular_rate*np.ones(len(neurons_ids))).astype(int)


# generate a spiketrain
fpga_spiketrain = generate_regular_rate_spiketrain(rates_vector, neurons_ids)


# plot the prepared spiketrain to verify it is correct
plt.figure(figsize=(18,2), dpi=150)
plt.plot(fpga_spiketrain[:,0], fpga_spiketrain[:,1], '.')
plt.xlabel("time (s)")
plt.ylabel("spikegen")

In [ ]:
fpga_spike_gen = model.get_fpga_spike_gen()
target_chip = chip_id # should be the same where _physical_neurons are located
fpga_spike_gen.stop()

ut.set_fpga_spike_gen(fpga_spike_gen,
                      spike_times=fpga_spiketrain[:,0],
                      indices=fpga_spiketrain[:,1].astype(int),
                      target_chips=[target_chip]*len(fpga_spiketrain),
                      isi_base=900,
                      repeat_mode=True)

# start the playback of the uploaded spike train
fpga_spike_gen.start()

### **FF Curve Measuremnt**

In [ ]:
# Example F-F curve measurement
recording_length_s = 1

rate_min = 10 #Hz
rate_max = 400 #Hz
rate_step = 10 #Hz
recorded_rates = []

for rate in np.arange(rate_min, rate_max, rate_step):
    
    rates_vector = (rate*np.ones(len(neurons_ids))).astype(int)
    
    # prepare the spiketrain
    fpga_spiketrain = generate_regular_rate_spiketrain(rates_vector, neurons_ids)
    
    # stop the spikegen while it is being reset
    fpga_spike_gen.stop()
    ut.set_fpga_spike_gen(fpga_spike_gen,
                      spike_times=fpga_spiketrain[:,0],
                      indices=fpga_spiketrain[:,1].astype(int),
                      target_chips=[target_chip]*len(fpga_spiketrain),
                      isi_base=900,
                      repeat_mode=True)
    
    # start the spikegen with the new spiketrain
    fpga_spike_gen.start()
    
    # record and sort events
    sink_node.get_events() # "reset", simply empties the filter
    time.sleep(recording_length_s) # physically wait for the set amount of time
    events = sink_node.get_events()
    
    avg_firing_rate = len(events)/(len(neurons_ids)*recording_length_s)
    print(f"Input rate: {rate} Hz")
    print(f"Average firing rate: {avg_firing_rate} Hz")
    
    rates = dict.fromkeys(neurons_ids, 0)
    if len(events) > 0:
        for evt in events:
            rates[evt.neuron_id]+=1
        # convert event counts to individual rates by dividing them
        # by the time interval
        for n_id, evt_count in rates.items():
            rates[n_id]=evt_count/recording_length_s
            
    recorded_rates.append(rates)
    

In [ ]:
## store and plot the firing rate recordings

x_labels = np.arange(rate_min, rate_max, rate_step)
mean_y_labels = []
std_y_labels = []

for line, fine in zip(recorded_rates, x_labels):
    rates = np.fromiter(line.values(), dtype=float)
    #print(line.values())
    mean_rate = rates.mean()
    std_rate = rates.std()
    mean_y_labels.append(mean_rate)
    std_y_labels.append(std_rate)
    
plt.figure()
plt.plot(x_labels, mean_y_labels)
plt.fill_between(x_labels,np.array(mean_y_labels)- np.array(std_y_labels), np.array(mean_y_labels) + np.array(std_y_labels), alpha = 0.3, color = 'red')
plt.xlabel('Input firing rate (Hz)')
plt.ylabel('Mean Firing Rate')